In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/sql-murder-mystery.db")


## Paso 1: Reporte del crimen

Se consulta la tabla `crime_scene_report` para encontrar el reporte del:

- Tipo: asesinato
- Fecha : ocurrido el 15 de enero de 2018 
- Ciudad: SQL City.


In [11]:
query = """
SELECT *
FROM crime_scene_report
WHERE type = 'murder'
  AND date = 20180115
  AND city = 'SQL City'
"""

df = pd.read_sql(query, conn)
df

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


El resultado muestra el reporte de un crimen y nos da las sigueintes pistas:

- Se identifican dos testigos.
2. uno vive en la ultima casa de Northestern DR.
3. Otro se llama Annabel y vive en Franklin ave.

## Paso 2: Identificación del primer testigo

A partir del reporte del crimen, se busca a la persona que vive en la última casa de Northwestern Dr.

In [22]:
query = """
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
"""

df = pd.read_sql(query,conn)
df

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
1,17729,Lasonya Wildey,439686,3824,Northwestern Dr,917817122
2,53890,Sophie Tiberio,957671,3755,Northwestern Dr,442830147
3,73368,Torie Thalmann,773862,3697,Northwestern Dr,341559436
4,96595,Coretta Cubie,303645,3631,Northwestern Dr,378403829
5,19420,Cody Schiel,890431,3524,Northwestern Dr,947110049
6,93509,Emmitt Aceuedo,916706,3491,Northwestern Dr,979073160
7,87456,Leonora Wolfsberger,215868,3483,Northwestern Dr,565203106
8,36378,Freddie Ellzey,267882,3449,Northwestern Dr,474117596
9,53076,Boris Bijou,664914,3327,Northwestern Dr,401191868


Resultado Muestra de busqueda de direccion: 

- Se ordenan las direcciones de forma descendente para encontrar la última casa de la calle.
- la ultima casa corresponde el 4919 ya que corresponde al numero mas alto que aparece en la primera fila. 
- El testigo identificado es Morty Schapiro.


# Paso 3: Entrevista al testigo

- Una vez identificado el primer testigo, consultamos la tabla `interview` usando su `person_id` para conocer su declaración.

In [28]:
query = """
SELECT *
FROM interview
WHERE person_id = 14887

"""
df = pd.read_sql(query, conn)

person_id = df["person_id"][0]
transcript = df["transcript"][0]

print(f" Person ID: {person_id}\n")
print("Declaración del testigo:\n")
print(transcript)



 Person ID: 14887

Declaración del testigo:

I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".


- Resultado muestra Testigo:

el declara que escucho un dispaso y que vio salir un hombre corriendo con una bolsa de Get Fit now Gym.

pistas claves:

- El numero de membresía espezaba por 48Z
- solo los miembros Gold tienen esa bolsa
- El hombre se fue en un coche con un numero de placa que tiene H42W


# Paso 3: Buscar sospechoso en el GYM

- A partir de la declaración del testigo, se buscan los miembros del gimnasio Get Fit Now cuyo número de membresía comienza por "48Z" y tienen estatus "gold".

In [ ]:
query = """
SELECT *
FROM get_fit_now_member
WHERE membership_status = 'gold'
  AND id LIKE '48Z%'
  
"""

df = pd.read_sql(query, conn)
df

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


- Resultado Muestra del listado de sospechosos GYM.

Se identifican dos posibles sospechosos cuyos, Id y membresias cumplen con las condiciones:

- 48Z7A Joe Germuska
- 48Z55 Jeremy Bowers


# Paso 5: Cruzar la matricula de los sospechosos.

A partir de los sospechosos encontrados en el gimnasio, se cruza la tabla `person` con `drivers_license` para identificar cuál de ellos tiene una matrícula que contiene "H42W".


In [30]:
query = """
SELECT
    p.id,
    p.name,
    dl.plate_number
FROM person AS p
JOIN drivers_license AS dl
    ON p.license_id = dl.id
WHERE p.id IN (28819 , 67318)
"""
df = pd.read_sql(query, conn)
df


,id,name,plate_number
0,67318,Jeremy Bowers,0H42W2


- Resultado Muestra del cruce 

- Al cruzar los sospechosos con sus licencias de conducción, se observa que:

- Jeremy Bowers tiene una matrícula que contiene "H42W".

Por lo tanto, Jeremy Bowers es identificado como el principal sospechoso.



# Paso 6 : Entrevista del asesino

Una vez identificado el asesino, se consulta la tabla ´interview` para conocer su declaración y obtener nuevas pistas de quien lo contrato. 

In [31]:
query = """
SELECT * 
FROM interview
WHERE person_id = 67318

"""
df = pd.read_sql(query, conn)

person_id = df["person_id"][0]
transcript = df["transcript"][0]

print(f" Person ID: {person_id}\n")
print(" Declaración del asesino:\n")
print(transcript)

 Person ID: 67318

 Declaración del asesino:

I was hired by a woman with a lot of money. I don't know her name but I know she's around 5'5" (65") or 5'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.



- Resultado de la Entrevista del Asesino:

declara que fue contratado por una mujer con mucho dinero:

- no sabe su nombre pero:
- mide aproximadamente 65 y 67 pulgadas( 1,65 m a 1,70 m)
- Tiene cabello color rojo
- Conduce un coche Tesla Model S.
- Asistio al concierto "SQL symphony" tres veces en dic de 2017


# Paso 7: Identificación de la autora intelectual

A partir de la declaración del asesino, se buscan personas que coincidan con el perfil descrito: 
- Mujer 
- Pelo rojo
- Altura 65 y 67 pulgadas( 1,65 m a 1,70 m)
- Coche Tesla Model S
- Asistente al concierto "SQL symphony" tres veces en dic de 2017

In [34]:
query= """
SELECT
    p.id,
    p.name,
    dl.height,
    dl.hair_color,
    dl.car_make,
    dl.car_model,
    COUNT(f.event_name) AS concert_attendance
FROM person AS p
JOIN drivers_license AS dl
    ON p.license_id = dl.id
JOIN facebook_event_checkin AS f
    ON p.id = f.person_id
WHERE dl.gender = 'female'
  AND dl.hair_color = 'red'
  AND dl.height > 60
  AND dl.car_make = 'Tesla'
  AND dl.car_model = 'Model S'
  AND f.event_name = 'SQL Symphony Concert'
GROUP BY p.id, p.name, dl.height, dl.hair_color, dl.car_make, dl.car_model
HAVING COUNT(f.event_name) = 3

"""
df = pd.read_sql(query, conn)
df

,id,name,height,hair_color,car_make,car_model,concert_attendance
0,99716,Miranda Priestly,66,red,Tesla,Model S,3


### Conclusión: 

- Tras analizar las pistas se identifico que: 

- Asesino Jeremy Bowers.
- Autora Intectual del crimen es Miranda Priestly. 
- No se encontro registro de entrevista a Miranda Priestly en la tabla 'interview'.
- Sin, embargo a partir del cruce de datos y las pistas entregadas, se confirma su implicación en el caso. 



